In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

df = pd.read_csv("appointments_clean.csv")

feature_cols = [
    "age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
    "reminder_sent", "booking_channel", "visit_type"
]

X = df[feature_cols].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], dtype=int)
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_acc = (y_test == 0).mean()
print("Feature count:", X.shape[1])
print("Feature columns:", list(X.columns))
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("No-shows in test set:", y_test.sum())
print(f"Baseline accuracy (always predict Attended): {baseline_acc:.4f} ({(y_test == 0).sum()} of {len(y_test)})")

Feature count: 11
Feature columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train rows: 792 | Test rows: 198
No-shows in test set: 49
Baseline accuracy (always predict Attended): 0.7525 (149 of 198)


In [2]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_preds = lr.predict(X_test_s)
lr_probs = lr.predict_proba(X_test_s)[:, 1]

lr_acc = accuracy_score(y_test, lr_preds)
lr_prec = precision_score(y_test, lr_preds)
lr_rec = recall_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds)
lr_auc = roc_auc_score(y_test, lr_probs)

print(f"Accuracy : {lr_acc:.4f}")
print(f"Precision: {lr_prec:.4f}")
print(f"Recall   : {lr_rec:.4f}")
print(f"F1-score : {lr_f1:.4f}")
print(f"ROC-AUC  : {lr_auc:.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_preds))

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef_scaled": lr.coef_[0],
    "scale_sd": scaler.scale_
})
print("\nScaled coefficients:\n", coef_df.to_string(index=False))

rem_idx = list(X.columns).index("reminder_sent")
rem_coef = lr.coef_[0][rem_idx]
rem_scale = scaler.scale_[rem_idx]
or_scaled = np.exp(rem_coef)
or_real = np.exp(rem_coef / rem_scale)

print(f"\nreminder_sent scaled coef: {rem_coef:.4f}, SD (scale): {rem_scale:.4f}")
print(f"np.exp(coef) [per 1 SD]: {or_scaled:.4f}")
print(f"np.exp(coef / scale) [No=0 to Yes=1]: {or_real:.4f}")

Accuracy : 0.7879
Precision: 0.6296
Recall   : 0.3469
F1-score : 0.4474
ROC-AUC  : 0.7889
Confusion Matrix:
 [[139  10]
 [ 32  17]]

Scaled coefficients:
                 feature  coef_scaled   scale_sd
                    age    -0.253880  16.655863
            distance_km     0.253448   6.051810
                fee_inr     0.452761 184.253540
              lead_days     0.614471   8.614206
         prior_no_shows     0.649612   0.968578
          reminder_sent    -0.580669   0.494975
    booking_channel_App    -0.046804   0.486016
  booking_channel_Phone     0.215881   0.481046
booking_channel_Walk-In    -0.186364   0.435178
visit_type_Consultation     0.015658   0.492792
   visit_type_Follow-up    -0.015658   0.492792

reminder_sent scaled coef: -0.5807, SD (scale): 0.4950
np.exp(coef) [per 1 SD]: 0.5595
np.exp(coef / scale) [No=0 to Yes=1]: 0.3094


In [3]:
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
dt_full_train_acc = dt_full.score(X_train, y_train)
dt_full_test_acc = dt_full.score(X_test, y_test)
print(f"Full Decision Tree (depth={dt_full.get_depth()}): Train Acc={dt_full_train_acc:.4f}, Test Acc={dt_full_test_acc:.4f}")

dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)
dt_3_train_acc = dt_3.score(X_train, y_train)
dt_3_test_acc = dt_3.score(X_test, y_test)
dt_3_preds = dt_3.predict(X_test)
dt_3_probs = dt_3.predict_proba(X_test)[:, 1]
print(f"Depth-3 Decision Tree: Train Acc={dt_3_train_acc:.4f}, Test Acc={dt_3_test_acc:.4f}")

Full Decision Tree (depth=15): Train Acc=1.0000, Test Acc=0.6717
Depth-3 Decision Tree: Train Acc=0.8220, Test Acc=0.7778


In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train_acc = rf.score(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]
rf_test_acc = accuracy_score(y_test, rf_preds)
rf_rec = recall_score(y_test, rf_preds)
rf_auc = roc_auc_score(y_test, rf_probs)

print(f"Random Forest: Train Acc={rf_train_acc:.4f}, Test Acc={rf_test_acc:.4f}, Recall={rf_rec:.4f}, ROC-AUC={rf_auc:.4f}")

imp_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
print("\nTop 3 Random Forest feature importances:")
print(imp_df.head(3).to_string(index=False))

comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": round(lr_acc, 4),
        "Recall": round(lr_rec, 4),
        "ROC-AUC": round(lr_auc, 4),
    },
    {
        "Model": "Decision Tree (max_depth=3)",
        "Accuracy": round(dt_3_test_acc, 4),
        "Recall": round(recall_score(y_test, dt_3_preds), 4),
        "ROC-AUC": round(roc_auc_score(y_test, dt_3_probs), 4),
    },
    {
        "Model": "Random Forest (100 trees)",
        "Accuracy": round(rf_test_acc, 4),
        "Recall": round(rf_rec, 4),
        "ROC-AUC": round(rf_auc, 4),
    },
])
print("\nTest-set comparison table (for reporting):")
print(comparison_df.to_string(index=False))

Random Forest: Train Acc=1.0000, Test Acc=0.7475, Recall=0.3061, ROC-AUC=0.7057

Top 3 Random Forest feature importances:
    feature  importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739

Test-set comparison table (for reporting):
                      Model  Accuracy  Recall  ROC-AUC
        Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
  Random Forest (100 trees)    0.7475  0.3061   0.7057


In [5]:
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc",
)
grid.fit(X_train, y_train)
print("Tuned RF best_params_:", grid.best_params_)
print(f"Tuned RF best_score_ (CV ROC-AUC): {grid.best_score_:.4f}")

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

cv_scores = {
    "Logistic Regression (Pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision Tree (max_depth=3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random Forest (default)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random Forest (tuned)": grid.best_score_,
}

print("\n5-fold CV ROC-AUC on training set:")
for name, score in cv_scores.items():
    print(f"  {name}: {score:.4f}")

oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("\nOut-of-fold threshold comparison (training set):")
for thresh in [0.5, 0.35]:
    oof_preds = (oof >= thresh).astype(int)
    p = precision_score(y_train, oof_preds)
    r = recall_score(y_train, oof_preds)
    f = f1_score(y_train, oof_preds)
    flagged = oof_preds.sum()
    print(f"  Threshold {thresh}: Precision={p:.4f}, Recall={r:.4f}, F1={f:.4f}, Flagged={flagged}")

chosen_threshold = 0.35
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
final_preds = (probs >= chosen_threshold).astype(int)

print("\nFinal single test-set evaluation (Logistic Regression @ 0.35):")
print(f"  Precision: {precision_score(y_test, final_preds):.4f}")
print(f"  Recall   : {recall_score(y_test, final_preds):.4f}")
print(f"  Flagged  : {final_preds.sum()} of {len(y_test)}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, probs):.4f}")

risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": final_preds,
})
risk_scores.to_csv("test_risk_scores.csv", index=False)
print("\nSaved test_risk_scores.csv with shape:", risk_scores.shape)
print("Rows flagged in CSV:", risk_scores["flag"].sum())

Tuned RF best_params_: {'max_depth': 3, 'n_estimators': 200}
Tuned RF best_score_ (CV ROC-AUC): 0.7673

5-fold CV ROC-AUC on training set:
  Logistic Regression (Pipeline): 0.7737
  Decision Tree (max_depth=3): 0.6784
  Random Forest (default): 0.7466
  Random Forest (tuned): 0.7673

Out-of-fold threshold comparison (training set):
  Threshold 0.5: Precision=0.5753, Recall=0.2485, F1=0.3471, Flagged=73
  Threshold 0.35: Precision=0.5130, Recall=0.4675, F1=0.4892, Flagged=154

Final single test-set evaluation (Logistic Regression @ 0.35):
  Precision: 0.5714
  Recall   : 0.5714
  Flagged  : 49 of 198
  ROC-AUC  : 0.7889

Saved test_risk_scores.csv with shape: (198, 4)
Rows flagged in CSV: 49
